In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 10)

driver.get("http://localhost:5173/")
driver.execute_script("window.localStorage.clear(); window.sessionStorage.clear();")
driver.get("http://localhost:5173/login")

wait.until(EC.presence_of_element_located((By.ID, "username")))
driver.find_element(By.ID, "username").send_keys("shawon@gmail.com")
driver.find_element(By.ID, "password").send_keys("12345678")
driver.find_element(By.ID, "sign-in-btn").click()

time.sleep(3)
print("URL:", driver.current_url)
print("Page text:", driver.find_element(By.TAG_NAME, "body").text[:200])

In [ ]:
try:
    # Go to POS / Sales (sidebar button, verified in AppShell.jsx)
    wait.until(EC.element_to_be_clickable((By.XPATH, "//aside//button[contains(., 'POS / Sales')]"))).click()
    wait.until(EC.visibility_of_element_located((By.XPATH, "//h2[text()='POS / Sales']")))

    # Add the first available medicine to the cart (Add button, verified in CatalogTable.jsx)
    wait.until(EC.presence_of_element_located((By.XPATH, "//tr[contains(@class, 'pos-row')]")))
    add_buttons = [b for b in driver.find_elements(By.XPATH, "//tr[contains(@class, 'pos-row')]//button[contains(., 'Add')]") if b.is_displayed() and b.is_enabled()]
    assert add_buttons, "No addable medicine found in the catalog."
    med_name = add_buttons[0].find_element(By.XPATH, "./ancestor::tr[1]").text.split("\n")[0]
    print("Medicine:", med_name)
    add_buttons[0].click()
    time.sleep(2)

    # Verify the cart has the item
    body = driver.find_element(By.TAG_NAME, "body").text
    assert "Current Sale" in body and "Cart is empty" not in body, "Cart has no item."
    assert med_name in body, "Added medicine not shown in the cart."

    # Complete the sale with the default Cash method (verified in PaymentCard.jsx)
    wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Complete Sale') and not(contains(., 'Reviewed'))]"))).click()
    time.sleep(3)
    approve = [b for b in driver.find_elements(By.XPATH, "//button[contains(., 'Reviewed')]") if b.is_displayed()]
    if approve:
        print("Approval modal appeared, confirming...")
        approve[0].click()
        time.sleep(3)

    # Verify the real success behavior: receipt modal with "Sale Completed" + invoice
    wait.until(EC.visibility_of_element_located((By.XPATH, "//*[text()='Sale Completed']")))
    invoice = driver.find_element(By.XPATH, "//*[contains(text(), 'Invoice #')]").text
    print("Confirmation:", "Sale Completed /", invoice)
    print("Current URL:", driver.current_url)
    print("PASS: Checkout")
except Exception as e:
    print("FAIL: Checkout")
    print("Error:", e)
    driver.save_screenshot("18_checkout_FAIL.png")

In [ ]:
driver.quit()